# LoCED × MaxText v0.2.5 — 500M on 2×T4 with HF streaming data

This is the **GPU training test** for the ~500.23M-parameter LoCED model on
Kaggle **2× Tesla T4**. It streams the mixed `nampdn-ai` datasets through the
Hugging Face token, tokenizes with a 32k-vocab tokenizer, and runs MaxText Muon
with FSDP=2.

### What it does

- reads `HF_TOKEN` from a **Kaggle secret** (never hard-coded here);
- checks gated-dataset access;
- builds the 500M model with `d=1024`, 8 routed + 1 shared expert, vocab 32k;
- trains for `TRAIN_STEPS` steps (default 20) on a mixed stream of:
  `tiny-textbooks`, `tiny-strange-textbooks`, `tiny-codes`,
  `tiny-math-textbooks`, `tiny-orca-textbooks`, `tiny-webtext`;
- writes Orbax checkpoints to `/kaggle/working/loced_runs/...`;
- `hf_train.py` checkpoints every 10 minutes (`LOCED_CKPT_AFTER_S=0`).

### Before running

1. Kaggle → **Add-ons → Secrets** → add `HF_TOKEN` (the rotated token).
2. Kaggle → **Accelerator: GPU T4 ×2**.
3. Run all. The first install/compile is slow (~15–25 min). For a short test
   keep `TRAIN_STEPS=20`; for a longer run raise it.

This is a GPU training test, not the 8-chip TPU full run. It does not embed or
print the token.


In [ ]:
# ============================================================================
# LoCED x MaxText v0.2.5 - 500M 2x T4 HF training
# ============================================================================
import json
import os
import shutil
import subprocess
import sys
import textwrap
import time
from pathlib import Path

# ---------------------------------------------------------------------------
# User knobs
# ---------------------------------------------------------------------------
ACCELERATOR = os.environ.get("LOCED_ACCELERATOR", "gpu")  # "gpu" now, "tpu" later
MAXTEXT_REF = "maxtext-v0.2.5"
MAXTEXT_REPO = "https://github.com/AI-Hypercomputer/maxtext.git"
MAXTEXT_EXTRAS = "cuda12" if ACCELERATOR == "gpu" else "tpu"
MODEL_CONFIG = "loced/maxtext/configs/loced_500m_t4x2.yml"
TRAIN_STEPS = int(os.environ.get("LOCED_TRAIN_STEPS", "20"))
PER_DEVICE_BATCH = int(os.environ.get("LOCED_BATCH", "1"))
SEQ_LEN = int(os.environ.get("LOCED_SEQ", "64"))
RUN_NAME = "loced-500m-t4x2-hf"
HF_DATASETS = "nampdn-ai/tiny-textbooks,nampdn-ai/tiny-strange-textbooks,nampdn-ai/tiny-codes,nampdn-ai/tiny-math-textbooks,nampdn-ai/tiny-orca-textbooks,nampdn-ai/tiny-webtext"
HF_RATIOS = "2.7,2.7,1.6,0.635,1.0,4.5"
HF_TOKENIZER = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# The notebook clones this repo if LoCED is not already present.
LOCED_REPO_URL = os.environ.get(
    "LOCED_REPO_URL", "https://github.com/offx-zinth/loced.git"
)
LOCED_REPO_REF = os.environ.get("LOCED_REPO_REF", "master")

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
WORK.mkdir(parents=True, exist_ok=True)
MAXTEXT_DIR = WORK / "maxtext-v0.2.5"
VENV_DIR = WORK / "loced-venv"
PY = VENV_DIR / "bin" / "python"
SCRIPTS = WORK / "loced_scripts"
RUNS = WORK / "loced_runs"
CACHE = WORK / "jax_cache"
HF_CACHE = WORK / "hf_cache"
CHECKPOINT_DIR = RUNS / RUN_NAME / "checkpoints"
for _p in (SCRIPTS, RUNS, CACHE, HF_CACHE, CHECKPOINT_DIR):
    _p.mkdir(parents=True, exist_ok=True)


def _loced_parent(root: Path):
    # Return the directory to put on PYTHONPATH for `import loced`.
    if (root / "loced" / "maxtext" / "train.py").is_file():
        return root
    # offx-zinth/loced is the package contents at the repo root.
    if root.name == "loced" and (root / "maxtext" / "train.py").is_file():
        return root.parent
    return None


def _find_loced_root():
    candidates = []
    if os.environ.get("LOCED_ROOT"):
        candidates.append(Path(os.environ["LOCED_ROOT"]))
    candidates += [
        WORK,
        WORK / "loced",
        Path.cwd(),
        Path.cwd() / "loced",
        Path("/kaggle/input"),
    ]
    for root in candidates:
        found = _loced_parent(root)
        if found is not None:
            return found
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for p in kaggle_input.glob("**/maxtext/train.py"):
            repo = p.parents[1]
            if repo.name == "loced" or (
                (repo / "attention.py").is_file()
                and (repo / "config.py").is_file()
            ):
                return repo.parent
        for p in kaggle_input.glob("**/loced/maxtext/train.py"):
            return p.parents[2]
    return None


# Keep a reused Kaggle working directory in sync with the branch under test.
_target = WORK / "loced"
if LOCED_REPO_URL and (_target / ".git").is_dir():
    _env = os.environ.copy()
    _env["GIT_TERMINAL_PROMPT"] = "0"
    print(f"updating existing checkout {_target} ({LOCED_REPO_REF})")
    subprocess.run(
        ["git", "-C", str(_target), "fetch", "--depth", "1",
         "origin", LOCED_REPO_REF],
        check=True, env=_env,
    )
    subprocess.run(
        ["git", "-C", str(_target), "reset", "--hard", "FETCH_HEAD"],
        check=True, env=_env,
    )

LOCED_ROOT = _find_loced_root()
if LOCED_ROOT is None and LOCED_REPO_URL:
    target = WORK / "loced"
    if not (target / ".git").is_dir():
        if target.is_dir() and any(target.iterdir()):
            print(f"{target} exists without .git; treating it as the source")
        else:
            env = os.environ.copy()
            env["GIT_TERMINAL_PROMPT"] = "0"
            print(f"cloning {LOCED_REPO_URL} ({LOCED_REPO_REF}) -> {target}")
            subprocess.run(
                ["git", "clone", "--depth", "1", "--branch", LOCED_REPO_REF,
                 LOCED_REPO_URL, str(target)],
                check=True, env=env,
            )
    LOCED_ROOT = _find_loced_root()

print(f"ACCELERATOR : {ACCELERATOR}")
print(f"WORK        : {WORK}")
print(f"LOCED_ROOT  : {LOCED_ROOT}")
print(f"MAXTEXT_DIR : {MAXTEXT_DIR}")
print(f"VENV        : {VENV_DIR}")
if LOCED_ROOT is None:
    raise RuntimeError(textwrap.dedent('''
        LoCED source not found.
        - Upload the repository folder (containing `loced/`) to
          /kaggle/working/loced, or as a Kaggle Dataset under
          /kaggle/input/...
        - Or set LOCED_REPO_URL (and LOCED_REPO_REF) in this cell to a branch
          that contains `loced/`.
    '''))
print("nvidia-smi:")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "-L"], check=False)
else:
    print("nvidia-smi not found (CPU-only notebook kernel?)")

## 1. Python 3.12 + MaxText v0.2.5 (CUDA 12)

The install adds `datasets`, `transformers`, and `huggingface_hub` so the
streaming tokenizer and gated-dataset loader are available inside the venv.


In [ ]:
SKIP_INSTALL = os.environ.get("LOCED_SKIP_INSTALL", "0") == "1"

if SKIP_INSTALL:
    print("LOCED_SKIP_INSTALL=1: using existing venv/MaxText checkout")
else:
    if shutil.which("uv") is None:
        print("installing uv ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    UV = shutil.which("uv")
    if UV is None:
        raise RuntimeError("uv is not on PATH after pip install")

    if not PY.exists():
        print("creating Python 3.12 venv ...")
        subprocess.run(
            [UV, "venv", "--python", "3.12", "--seed", str(VENV_DIR)],
            check=True,
        )
    print(subprocess.check_output([str(PY), "--version"], text=True).strip())

    if not (MAXTEXT_DIR / "pyproject.toml").is_file():
        print(f"cloning MaxText {MAXTEXT_REF} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", MAXTEXT_REF,
             MAXTEXT_REPO, str(MAXTEXT_DIR)],
            check=True,
        )
    else:
        print("MaxText checkout already present")

    print(f"installing maxtext[{MAXTEXT_EXTRAS}] (resolution=lowest) ...")
    subprocess.run(
        [UV, "pip", "install", "--python", str(PY), "--link-mode=copy",
         "-e", f".[{MAXTEXT_EXTRAS}]", "--resolution=lowest"],
        cwd=str(MAXTEXT_DIR), check=True,
    )
    subprocess.run(
        [UV, "pip", "install", "--python", str(PY), "--link-mode=copy",
         "pytest", "pytest-timeout",
         "datasets>=2.19,<4", "transformers>=4.40,<5", "huggingface_hub>=0.23,<1"],
        check=True,
    )
    print("dependency check:")
    subprocess.run([UV, "pip", "check", "--python", str(PY)], check=False)

print(subprocess.check_output(
    [str(PY), "-c",
     "import sys, maxtext; print(sys.version); print(maxtext.__file__)"],
    text=True,
))

## 2. GPU + real-MaxText sanity check


In [ ]:
BASE_ENV = os.environ.copy()
BASE_ENV.update({
    "PYTHONPATH": f"{MAXTEXT_DIR / 'src'}:{LOCED_ROOT}",
    "JAX_COMPILATION_CACHE_DIR": str(CACHE),
    "XLA_PYTHON_CLIENT_MEM_FRACTION": "0.90",
    "XLA_PYTHON_CLIENT_PREALLOCATE": "true",
    "TOKENIZERS_PARALLELISM": "false",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    "PYTHONDONTWRITEBYTECODE": "1",
    "TF_CPP_MIN_LOG_LEVEL": "2",
})

# jax-cuda12 loads the pip-installed NVIDIA libraries with ctypes.  In a venv
# they are under site-packages/nvidia/*/lib, which is not on the default loader
# path, so cuSPARSE/cuDNN can otherwise be reported as "not found".
_nvidia_code = (
    "import site; from pathlib import Path; "
    "print(':'.join(str(p) for r in site.getsitepackages() "
    "for p in sorted((Path(r) / 'nvidia').glob('*/lib')) if p.is_dir()))"
)
_nvidia_libs = subprocess.check_output([str(PY), "-c", _nvidia_code], text=True).strip()
if _nvidia_libs:
    BASE_ENV["LD_LIBRARY_PATH"] = ":".join(
        x for x in (_nvidia_libs, os.environ.get("LD_LIBRARY_PATH", "")) if x
    )
print("NVIDIA lib dirs:", len(_nvidia_libs.split(":")) if _nvidia_libs else 0)


def run_script(name, code, timeout=3600, extra_env=None, check=True, cwd=None):
    path = SCRIPTS / name
    path.write_text(textwrap.dedent(code))
    env = BASE_ENV.copy()
    if extra_env:
        env.update({str(k): str(v) for k, v in extra_env.items()})
    t0 = time.time()
    proc = subprocess.run(
        [str(PY), str(path)],
        cwd=str(cwd or LOCED_ROOT),
        env=env,
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    elapsed = time.time() - t0
    print(proc.stdout)
    if proc.stderr.strip():
        print("[stderr]\n" + proc.stderr[-12000:])
    print(f"[{name}] exit={proc.returncode} elapsed={elapsed:.1f}s")
    if check and proc.returncode != 0:
        raise RuntimeError(f"{name} failed (exit {proc.returncode})")
    return proc


gpu_check = r'''
import importlib.metadata
import os
import shutil
import subprocess
import sys

import jax
import maxtext

print("python  :", sys.version.split()[0])
print("jax     :", jax.__version__)
print("maxtext :", importlib.metadata.version("maxtext"))
print("backend :", jax.default_backend())
print("devices :", jax.devices())

expected = os.environ["LOCED_EXPECTED_BACKEND"]
assert jax.default_backend() == expected, (
    f"Expected {expected}, got {jax.default_backend()}. "
    "Set the Kaggle accelerator and restart the session."
)
assert len(jax.devices()) >= 1
for d in jax.devices():
    stats = d.memory_stats() or {}
    print(f"  {d}: {d.device_kind} bytes_limit={stats.get('bytes_limit')}")

# The GPU/TPU run must exercise the real MaxText utilities, not CPU stubs.
import maxtext.utils.maxtext_utils  # noqa: F401
import maxtext.utils.model_creation_utils  # noqa: F401
print("real MaxText utilities: OK")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
'''
run_script(
    "gpu_check.py", gpu_check,
    extra_env={"LOCED_EXPECTED_BACKEND": ACCELERATOR},
)

## 3. Hugging Face token + gated-dataset access

The token is read from the Kaggle secret `HF_TOKEN`. It is passed only through
the subprocess environment and is never written to the notebook or logs.


In [ ]:
import os

try:
    from kaggle_secrets import UserSecretClient
    HF_TOKEN = UserSecretClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN:
    raise RuntimeError(
        "Add HF_TOKEN as a Kaggle secret (Add-ons -> Secrets), then rerun. "
        "The token is not stored in this notebook."
    )

os.environ["HF_TOKEN"] = HF_TOKEN
print("HF token loaded from secret/env (value not printed).")

whoami_code = r'''
import os
from huggingface_hub import whoami
me = whoami(token=os.environ["HF_TOKEN"])
print("HF user:", me.get("name"))
print("HF token: OK")
'''
run_script("hf_whoami.py", whoami_code, extra_env={"HF_TOKEN": HF_TOKEN})


## 4. Gated-dataset access check

Streams one example from each dataset and prints its text column. This catches
a missing token, a not-accepted gated dataset, or a renamed column before the
500M model is built.


In [ ]:
data_check = r'''
import os

from datasets import load_dataset

names = os.environ["HF_DATASETS"].split(",")
for name in names:
    name = name.strip()
    ds = load_dataset(
        name, split="train", streaming=True,
        token=os.environ["HF_TOKEN"], trust_remote_code=True,
    )
    print(name, "features:", list(ds.features))
    example = next(iter(ds))
    text = None
    for col in ("text", "content", "body", "document", "prompt", "completion"):
        if col in example and isinstance(example[col], str):
            text = example[col]
            break
    if text is None:
        for value in example.values():
            if isinstance(value, str):
                text = value
                break
    print("  sample chars:", len(text) if isinstance(text, str) else type(text))
'''
run_script(
    "hf_data_check.py",
    data_check,
    timeout=1800,
    extra_env={
        "HF_TOKEN": HF_TOKEN,
        "HF_DATASETS": HF_DATASETS,
        "HF_HOME": str(HF_CACHE),
        "HF_DATASETS_CACHE": str(HF_CACHE / "datasets"),
    },
)


## 5. 500M HF streaming training (MaxText Muon, FSDP=2)

Builds the 500M model and trains for `TRAIN_STEPS` steps. The stream is mixed
using `HF_RATIOS`; `TinyLlama` has exactly 32,000 vocab entries, matching the
LoCED model.

For a longer run, change `TRAIN_STEPS` in the config cell. Checkpoints go to
`CHECKPOINT_DIR`; `hf_train.py` saves every 10 minutes from the start
(`LOCED_CKPT_AFTER_S=0`).


In [ ]:
train_code = f'''
from loced.maxtext.hf_train import main

main([
    "loced-500m-t4x2",
    "loced/maxtext/configs/loced_500m_t4x2.yml",
    "steps={TRAIN_STEPS}",
    "per_device_batch_size={PER_DEVICE_BATCH}",
    "max_target_length={SEQ_LEN}",
    "run_name={RUN_NAME}",
    "checkpoint_dir={CHECKPOINT_DIR}",
])
'''
run_script(
    "run_hf_train.py",
    train_code,
    timeout=7200,
    extra_env={
        "HF_TOKEN": HF_TOKEN,
        "HF_DATASETS": HF_DATASETS,
        "HF_RATIOS": HF_RATIOS,
        "HF_TOKENIZER": HF_TOKENIZER,
        "HF_HOME": str(HF_CACHE),
        "HF_DATASETS_CACHE": str(HF_CACHE / "datasets"),
        "LOCED_CKPT_AFTER_S": "0",
    },
)


## 6. Summary

If the cell above completed, the 500M model trained on the mixed gated HF
stream for `TRAIN_STEPS` steps on 2×T4 with MaxText Muon. The checkpoint is at
`CHECKPOINT_DIR`.

This is a GPU training test; the full TPU v5e-8 run is the separate
`LoCED_MaxText_500M_TPU8_Kaggle.ipynb` / `maxtext/hf_train.py` path.


In [ ]:
import os
from pathlib import Path

print("checkpoint dir:", CHECKPOINT_DIR)
if Path(CHECKPOINT_DIR).is_dir():
    files = sorted(Path(CHECKPOINT_DIR).rglob("*"))
    print("checkpoint files:", len(files))
    for p in files[:20]:
        print(" ", p.relative_to(CHECKPOINT_DIR), p.stat().st_size if p.is_file() else "dir")
else:
    print("no checkpoint found")
